# 01 · Análisis exploratorio de datos (EDA)

**Proyecto:** predicción de la deserción en el primer año de pregrado (Tec de Monterrey), reproduciendo y adaptando a Villar y Andrade (2024).

**Problema.** Al terminar el primer semestre, con la información de admisión y el desempeño en ese semestre, predecir qué estudiantes ya no estarán matriculados al cerrar su primer año. Población de estudio: cohortes AD19 y AD20 (la sección 2 explica por qué).

**Qué responde este notebook.** (1) Cómo se comporta la variable objetivo, (2) qué tan completos y confiables son los datos y qué variables no se pueden usar, (3) qué variables se relacionan con la deserción y en qué forma, y (4) qué decisiones de preprocesamiento se derivan de lo anterior.

**Reglas del análisis.**
- Se usa solo el **conjunto de desarrollo** (80 %). El conjunto de prueba sellado no se mira, ni siquiera de forma descriptiva.
- Solo se muestran **agregados** (tasas, conteos, cuantiles): los datos del TEC no pueden redistribuirse.
- La lógica vive en `src/desercion/eda.py` y las figuras en `src/desercion/plots.py`; este notebook ejecuta el paso `eda` y comenta los resultados. Requiere haber ejecutado `python -m desercion.cli prepare`.

In [1]:
import pandas as pd

from desercion import config, data
from desercion.cli import main

pd.set_option('display.width', 160)
pd.set_option('display.precision', 3)
main(['eda'])  # regenera tablas (reports/tables/eda) y figuras (reports/figures/eda)
T = config.path('tables_dir') / 'eda'

df = data.load_clean()
dev = data.study_frame(df, data.get_split(df), 'dev')
print(f'Población de estudio (desarrollo): {len(dev):,} estudiantes, '
      f'{dev[data.TARGET].mean():.2%} desertan')

Población de estudio (desarrollo): 19,408 estudiantes, 7.75% desertan


## 1. Variable objetivo

`desercion = 1 - retention`: el estudiante ya no está matriculado al terminar su primer año. El diccionario distingue cuatro momentos (`dropout.semester`): abandona *durante* el 1.er semestre (1), no se inscribe al 2.º (2), abandona durante el 2.º (3) o no se inscribe al 3.er semestre (4). La figura muestra todas las cohortes de pregrado como contexto.

![](../reports/figures/eda/eda01_desercion_momento.png)

In [2]:
pd.read_csv(T / 'objetivo_por_cohorte.csv', index_col=0)

,n,Durante el 1.er semestre,No se inscribe al 2.º semestre,Durante el 2.º semestre,No se inscribe al 3.er semestre,tasa_total
generation,,,,,,
AD14,8115,0.021,0.030,0.010,0.039,0.101
AD15,8033,0.046,0.000,0.010,0.036,0.092
AD16,8593,0.020,0.024,0.010,0.034,0.088
AD17,8630,0.022,0.023,0.011,0.031,0.087
AD18,9037,0.012,0.022,0.012,0.029,0.075
AD19,9759,0.007,0.048,0.007,0.033,0.094
AD20,9847,0.013,0.032,0.005,0.029,0.080


**Lectura.**
- **Desbalance severo y estable.** Deserta entre el 7,5 % y el 10,1 % de cada cohorte. En la población de estudio, el 7,8 %. Un clasificador que nunca alerta acierta el 92 % de las veces, así que la exactitud y el F1 ponderado no sirven para comparar modelos: se necesitan métricas centradas en la clase minoritaria (PR-AUC, *recall* y precisión).
- **La deserción se concentra en las transiciones**: la mayor parte ocurre cuando el estudiante no se inscribe al siguiente semestre (códigos 2 y 4), no a mitad del periodo.
- **Por qué se excluye el código 1.** Quien abandona *durante* el primer semestre ya no está matriculado cuando se hace la predicción, y su promedio del semestre refleja el abandono en lugar de anticiparlo. Incluirlo sería una fuga temporal.

## 2. ¿Por qué solo las cohortes AD19 y AD20?

Se considera ausente tanto el vacío (`NaN`) como el código textual `No information`. El mapa usa todo el pregrado de desarrollo para ver qué cohortes tienen cada variable.

![](../reports/figures/eda/eda02_faltantes_cohorte.png)

**Lectura.** Los faltantes son **estructurales: dependen de la cohorte**, no del estudiante.
- El desempeño del primer semestre (promedio, materias reprobadas y retiradas) solo existe en AD19 y AD20. Como es la información que el artículo identifica como la más predictiva, la población de estudio se restringe a esas dos cohortes: 19.408 estudiantes en desarrollo y 4.843 en prueba, muy por encima del mínimo de 5.000 registros.
- En esas cohortes casi todas las variables están completas. La educación de los padres y "primera generación", que faltan en más del 90 % de AD14–AD16, tienen menos del 5 % de faltantes en AD19–AD20.
- **El nivel socioeconómico y el rezago social siguen faltando en el 55 % y el 44 %** de los estudiantes, incluso en AD19–AD20 (ver la sección 3).

## 3. Calidad de los datos en la población de estudio

¿El dato faltante se asocia con desertar? Se compara la tasa de deserción con y sin dato (prueba chi-cuadrado).

![](../reports/figures/eda/eda03_faltantes_vs_desercion.png)

In [3]:
pd.read_csv(T / 'faltantes_vs_desercion.csv')

,variable,pct_ausente,tasa_con_dato,tasa_sin_dato,p_chi2,diferencia_pp
0,admission.test.online,97.063,0.133,0.076,6.515e-07,-5.748
1,socioeconomic.level,54.560,0.076,0.078,6.134e-01,0.205
2,social.lag,44.209,0.076,0.080,3.264e-01,0.390
3,admission.test,31.853,0.070,0.093,5.910e-08,2.246
4,admission.rubric,22.439,0.075,0.087,1.047e-02,1.193
5,zone.type,6.910,0.077,0.088,1.528e-01,1.122
6,mother.education.summary,3.988,0.077,0.092,1.506e-01,1.478
7,father.education.summary,3.102,0.078,0.065,2.662e-01,-1.317


**Lectura y decisiones.**
- **Nivel socioeconómico y rezago social: no utilizables.** Falta la mitad de los datos, la ausencia no depende del estudiante sino de que el dato no se registró, y la deserción es prácticamente la misma con y sin dato (entre 7,6 % y 8,0 %; *p* = 0,61 y 0,33). Cuando sí hay dato, apenas discriminan (AUC univariada de 0,51). Imputar la mitad de una variable sin señal solo añade ruido, así que **ambas se excluyen** del modelo.
- **Examen de admisión: dos escalas mezcladas.** El examen presencial va de 400 a 1600 y el examen en línea, de 0 a 100 (570 estudiantes, casi todos de AD20). En una sola columna, un puntaje en línea parecería extremadamente bajo. Se separaron en dos variables (`admission.test` y `admission.test.online`) en la limpieza de datos.
- **Faltantes informativos.** Quienes no tienen puntaje presencial desertan más (9,3 % frente a 7,0 %); quienes no tienen rúbrica de admisión, también (8,7 % frente a 7,5 %). Por eso las numéricas se imputan con la mediana **y** se añade un indicador de faltante (`SimpleImputer(add_indicator=True)`): la mediana sola borraría esa señal.

In [4]:
pd.read_csv(T / 'resumen_numericas.csv')

,variable,n_validos,min,p01,mediana,p99,max,mediana_retenidos,mediana_desertores
0,age,19408,16.00,17.000,18.00,21.00,54.00,18.000,18.000
1,scholarship.perc,19408,0.00,0.000,0.00,0.90,1.00,0.000,0.000
2,loan.perc,19408,0.00,0.000,0.00,0.30,0.45,0.000,0.000
3,PNA,19407,66.00,75.000,88.31,99.37,100.00,88.840,85.530
4,admission.test,13226,600.00,1010.000,1360.00,1550.00,1590.00,1370.000,1320.000
5,admission.test.online,570,1.00,42.000,67.00,89.31,94.00,67.000,66.000
6,admission.rubric,15053,0.00,19.000,36.00,50.00,50.00,36.000,36.000
7,general.math.eval,19408,0.00,20.250,50.00,98.50,100.00,50.000,48.084
8,english.evaluation,19408,1.00,1.000,6.00,7.00,7.00,6.000,6.000
9,FTE,19408,0.06,0.500,1.00,1.10,1.28,1.000,1.000


**Rangos y valores atípicos.**
- **Edad:** hasta 54 años, pero el 99 % tiene 21 o menos. Son casos reales; los árboles los manejan sin transformación y la regresión logística, tras estandarizar.
- **Promedio del 1.er semestre:** hay promedios de 0 (estudiantes que reprobaron todo; los que abandonaron a mitad de semestre ya se excluyeron). Son casos reales de alto riesgo y se conservan.
- **Carga académica (FTE):** el 1 % inferior cursa media carga o menos. Se conserva.

## 4. ¿Qué se relaciona con la deserción?

Tasa de deserción por deciles de cada variable numérica, con intervalo de confianza de Wilson al 95 % (banda). La línea punteada es la tasa global (7,8 %). La última es la variable construida **brecha de promedios** = promedio del 1.er semestre − promedio de bachillerato: mide cuánto cambia el rendimiento al pasar a la universidad.

![](../reports/figures/eda/eda04_perfil_numericas.png)

**Lectura de las numéricas.**
- **El promedio del 1.er semestre es la señal más fuerte y no es lineal:** el decil más bajo deserta un 21 %, pero por encima de 92 la tasa se estabiliza alrededor del 4,5 %. Un promedio bajo eleva mucho el riesgo; uno alto lo reduce poco.
- **El desempeño previo protege de forma monótona:** del 13 % al 4 % entre el decil más bajo y el más alto del promedio de bachillerato; el examen presencial repite el patrón (10 % → 4,6 %).
- **La brecha de promedios tiene forma de U:** quienes caen más de 7 puntos respecto al bachillerato desertan un 18 %, quienes mantienen su nivel un 5 %, y quienes mejoran mucho vuelven a subir levemente (8 %). Aporta algo que ninguno de los dos promedios captura por separado: el **cambio** de rendimiento.
- **Becas:** sin beca deserta el 10 %; con 40 % de beca o más, el 4 %.
- **Edad:** plana hasta los 18 años y creciente desde los 19; a los 20 o más, el 14 %.
- **Nota de matemáticas:** meseta entre 40 y 80 puntos, con riesgo alto solo en el decil inferior (12 %) y bajo en el superior (4,5 %). Relaciones así favorecen a los árboles.

![](../reports/figures/eda/eda05_perfil_categoricas.png)

**Lectura de las categóricas.**
- **Tipo de beca** es la categórica más discriminante: va del 0,3 % (*Leaders of Tomorrow*) al 10,3 % (sin beca). Las becas por talento académico desertan muy poco (4,2 %).
- **Programa:** del 6,5 % al 13,5 %. **Escuela:** del 6,5 % al 11,5 %. **Región del campus:** la región RM (6,3 %) deserta menos que las demás (7,5–10,6 %).
- **Educación del padre:** sin estudios universitarios, el 11,1 %; con maestría o doctorado, alrededor del 6 %. Es la señal socioeconómica que sí está disponible y completa.
- **Costo del colegio de origen:** casi plano (6,5–7,7 %), salvo la categoría *Not defined* (12,6 %).

## 5. ¿Cuánta señal aporta cada variable por sí sola?

La asociación de cada variable con la deserción se resume en una escala común: la AUC univariada reescalada como 2·|AUC − 0,5| (0 = ninguna señal, 1 = separación perfecta). En las categóricas se ordena por la tasa de cada categoría (medida descriptiva, en la misma muestra). En naranja, las variables del primer semestre.

![](../reports/figures/eda/eda06_asociacion_univariada.png)

In [5]:
pd.read_csv(T / 'asociacion_univariada.csv').head(12)

,variable,tipo,pct_ausente,auc,fuerza_auc,d_cohen,v_cramer
0,average.first.period,numérica,0.000,0.350,0.301,-0.534,NaN
1,PNA,numérica,0.005,0.391,0.218,-0.381,NaN
2,scholarship.perc,numérica,0.000,0.392,0.216,-0.403,NaN
3,scholarship.type,categórica,0.000,0.605,0.209,NaN,0.105
4,admission.test,numérica,31.853,0.405,0.190,-0.309,NaN
5,brecha.promedio,numérica,0.005,0.425,0.151,-0.395,NaN
6,admission.test.online,numérica,97.063,0.426,0.148,-0.222,NaN
7,father.education.summary,categórica,3.102,0.559,0.119,NaN,0.063
8,max.degree.parents,categórica,0.211,0.554,0.109,NaN,0.064
9,admission.rubric,numérica,22.439,0.448,0.105,-0.193,NaN


**Lectura.**
- **Ninguna variable es fuerte por sí sola.** La mejor, el promedio del 1.er semestre, tiene una AUC de 0,65; la mejor de ingreso, 0,61. La señal es débil y está repartida, por lo que no cabe esperar modelos con ROC-AUC muy por encima de 0,70.
- **El promedio del 1.er semestre supera a cualquier variable de ingreso**, como en el artículo.
- Género, procedencia, tipo de zona, colegio del Tec y carga académica casi no tienen señal univariada. Esto anticipa la selección de variables, que se decide con importancia por permutación dentro de la validación cruzada (`python -m desercion.cli select`).

## 6. Redundancia entre variables numéricas

![](../reports/figures/eda/eda07_correlacion.png)

**Lectura.**
- **% de beca y % total de apoyo** tienen ρ = 0,96: el total es la suma de beca y crédito. Esta dependencia exacta es la que produce los VIF de 10⁷ y la que llevó al artículo a descartar la regresión logística. Aquí se **excluye el total**, que no aporta información nueva, y la regresión logística se conserva.
- Las variables de desempeño previo (promedio de bachillerato, examen, beca y promedio del semestre) se correlacionan moderadamente entre sí (0,3–0,5): miden en parte lo mismo, así que su importancia en SHAP se reparte entre ellas.
- El resto de pares es débil (|ρ| < 0,3).

## 7. Del EDA a las decisiones de modelamiento

| Hallazgo | Decisión en el flujo | Dónde |
|---|---|---|
| El desempeño del 1.er semestre solo existe en AD19–AD20 | Población de estudio: AD19–AD20 | `data.study_frame` |
| Quien abandona durante el 1.er semestre tiene un promedio que refleja el abandono | Se excluye `dropout.semester == 1` | `data.study_frame` |
| 7,8 % de positivos | PR-AUC para elegir modelos, umbral que maximiza F2 y balanceo como hiperparámetro | `config.yaml`, `evaluate.py`, `models.py` |
| Nivel socioeconómico y rezago social: la mitad sin dato y sin señal | Se excluyen | `features.EXCLUDED` |
| Examen de admisión con dos escalas | Se separa en presencial y en línea | `data.clean` |
| Faltantes académicos informativos | Imputación por mediana **con indicador** | `features.make_preprocessor` |
| % total de apoyo = beca + crédito | Se excluye el total | `features.EXCLUDED` |
| El cambio de rendimiento tiene señal propia (forma de U) | Variable construida `brecha.promedio` | `features.add_engineered` |
| Relaciones no lineales y con mesetas | Se priorizan árboles y *boosting*; la regresión logística queda como referencia | `models.py` |
| Actividades extracurriculares medidas durante el año | Se excluyen (reflejan la permanencia en lugar de anticiparla) | `features.EXCLUDED` |
| Estudiantes con más de un registro | Partición agrupada por `student.id` | `data.make_split` |